# Stage 2 sampling: MATH, OlympiadBench, Omni-MATH

Samples 5 solutions for 300 problems per split with `Qwen/Qwen2.5-Math-1.5B-Instruct` (same model as the GSM8K run).

- **Runtime:** GPU (Runtime → Change runtime type → T4 or better).
- **Input:** `data/raw/processbench_<split>.jsonl` — the original problem text (the `pb_*` files strip `$...$`). Same ids and order as `pb_*`, so `--sample 300 --seed 0` picks the same problems from either file.
- **Output:** written straight to Google Drive, so a disconnect loses nothing. The script is resumable: if the session dies, run the setup cells again and re-run the split — finished problems are skipped.
- **Expected time:** about 55–65 min per split on a T4, about 3 h in total.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/hallucination_llm/stage2'
import os
os.makedirs(OUT, exist_ok=True)
print('outputs ->', OUT)

In [ ]:
%cd /content
!test -d hallucination_llm || git clone -b main https://github.com/amxndeep001/hallucination_llm.git
%cd /content/hallucination_llm
!git checkout main && git pull
!git log --oneline -1

Do not install requirements.txt on Colab: it pins torch 2.8.0, which breaks Colab's preinstalled CUDA libraries and torchvision. Stage 2 needs only torch (Colab's own), transformers and tqdm.

In [ ]:
!pip install -q transformers==4.57.6 tqdm
!python -c "import torch, transformers; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| transformers', transformers.__version__)"

In [ ]:
# Same settings for every split; the same seed picks the same problem positions in each file.
ARGS = ('--model Qwen/Qwen2.5-Math-1.5B-Instruct --sample 300 --seed 0 --max-new-tokens 1024 '
        '--n 5 --temperature 0.7 --top-p 0.95 --batch-size 2')
# Out of GPU memory? Use --batch-size 1 (slower).

In [ ]:
!python src/stage2_generate.py --input data/raw/processbench_math.jsonl --output {OUT}/samples_math.jsonl {ARGS}

In [ ]:
!python src/stage2_generate.py --input data/raw/processbench_olympiadbench.jsonl --output {OUT}/samples_olympiadbench.jsonl {ARGS}

In [ ]:
!python src/stage2_generate.py --input data/raw/processbench_omnimath.jsonl --output {OUT}/samples_omnimath.jsonl {ARGS}

In [ ]:
# Check each run, then copy the outputs into the repo's data/processed/ as well.
for split in ['math', 'olympiadbench', 'omnimath']:
    print(f'===== {split}')
    !python src/stage2_check.py {OUT}/samples_{split}.jsonl data/raw/processbench_{split}.jsonl
!cp {OUT}/samples_math.jsonl {OUT}/samples_olympiadbench.jsonl {OUT}/samples_omnimath.jsonl data/processed/
!ls -la {OUT} data/processed/samples_*.jsonl